# TauREx 3.0

## Setup

Lets setup the notebook. If the plots arent interactive then run this part again

In [1]:
import matplotlib.pyplot as plt
%matplotlib notebook
from ipywidgets import *
import numpy as np
import sys
import time

And lets disable logging

In [2]:
import taurex.log
taurex.log.disableLogging()

## Loading cross-sections

We need to point TauREx3 to our cross-sections. This is handled by the caching classes. Once a cross-section is loaded it does not need to be loaded again.
First lets import the classes:

In [3]:
from taurex.cache import OpacityCache,CIACache

Now lets point the xsection and cia cachers to our files:

In [4]:
OpacityCache().clear_cache()
OpacityCache().set_opacity_path("../../test_files/xsec/xsec_sampled_R15000_0.3-50")
CIACache().set_cia_path("../../test_files/cia/HITRAN/data")


TauREx3 is now ready to use them! For fun lets, try grabbing the H2O cross-section and plotting it. First tell the OpacityCache function to grab it.

In [5]:
h2o_xsec = OpacityCache()['H2O']

Numba not installed, using numpy instead


Now we can compute the cross-section for any pressure and temperature!
Lets try 2000K and 10 Pa.

In [6]:
h2o_xsec.opacity(2000, 10)

array([0.00000000e+00, 1.76839718e-27, 3.66508005e-27, ...,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00], shape=(76744,))

But why stop there? We can plot the temperature and pressure *interactively*

In [7]:
xsec_fig = plt.figure()
xsec_ax = xsec_fig.add_subplot(1,1,1)



xsec, = xsec_ax.plot(10000/h2o_xsec.wavenumberGrid,h2o_xsec.opacity(800,1e0))

def update_cross(temperature=1500.0,pressure=6.7):

    xsec.set_ydata(h2o_xsec.opacity(temperature,10**pressure))
    xsec_ax.relim();
    xsec_ax.autoscale_view()
    xsec_fig.canvas.draw()

interact(update_cross,temperature=(800.0,2000.0,100),pressure=(-1.0,10.0,1));

<IPython.core.display.Javascript object>

interactive(children=(FloatSlider(value=1500.0, description='temperature', max=2000.0, min=800.0, step=100.0),…

## Profiles

Now we need to setup our forward model. Lets create a temperature profile, we will use the Guillot profile but other brands are available:

In [8]:
from taurex.temperature import Guillot2010
guillot = Guillot2010(T_irr=1200.0)

Now lets do the same for our planet:

In [9]:
from taurex.planet import Planet
planet = Planet(planet_radius=1.0,planet_mass=1.0)

and the planets star:

In [10]:
from taurex.stellar import BlackbodyStar

star = BlackbodyStar(temperature=5700.0,radius=1.0)

Now we need to define a chemistry profile, first lets setup the chemical model, we're going for the free-type model so we'll use *TaurexChemistry* which allows us to freely add any molecule:

In [11]:
from taurex.chemistry import TaurexChemistry
chemistry = TaurexChemistry(fill_gases=['H2','He'],ratio=0.172)

#### Adding molecules

Now we need to add some molecules. This is accomplished by the *addGas* function. We can create varius types of gas profiles
for each molecule and add them in. Lets try the constant profile for H2O:

In [12]:
from taurex.chemistry import ConstantGas

h2o = ConstantGas('H2O',mix_ratio=1.2e-4)
chemistry.addGas(h2o)


We can also create the gas on the spot as well:

In [13]:
chemistry.addGas(ConstantGas('N2',mix_ratio=3.00739e-9))


And we're done for profiles! Like the cross-sections you can use them in isolation for your own evil deeds. Some require initialization with other profiles like pressure and altitude, you can find how to use them in the API documentation. An easy example are stars. Maybe you want to use the [**PHOENIX**](https://phoenix.ens-lyon.fr/Grids/BT-Settl/CIFIST2011_2015/FITS/) star library but don't want to bother coding the function to load and interpolate. Well you can just have *Taurex3* handle it for you!

In [14]:
from taurex.stellar import PhoenixStar
## 
anotherstar = PhoenixStar(phoenix_path='/path/to/phoenix/BT-Settl_M-0.0a+0.0',temperature=5200.0)
star_fig = plt.figure()
star_ax = star_fig.add_subplot(1,1,1)

star_wngrid = np.linspace(0,100000,1000)

anotherstar.initialize(star_wngrid)

pstar, = star_ax.plot(anotherstar.spectralEmissionDensity)

def update_cross(temperature=5200.0):
    anotherstar.temperature=temperature
    anotherstar.initialize(star_wngrid)
    
    pstar.set_ydata(anotherstar.spectralEmissionDensity)
    star_ax.relim();
    star_ax.autoscale_view()
    star_fig.canvas.draw()

interact(update_cross,temperature=(800.0,8000.0,400));


Exception: No file path or incorrect path to phoenix files defined - None

After that quick detour lets carry on with our goal.

## Building the model

Now we can build our transmission model! Lets first create our transmission model and add our profiles to them:


In [15]:
from taurex.model import TransmissionModel
tm = TransmissionModel(planet=planet,
                       temperature_profile=guillot,
                       chemistry=chemistry,
                       star=star,
                        atm_min_pressure=1e-0,
                       atm_max_pressure=1e6,
                       nlayers=30)

At this point our atmosphere has profiles but no physics! We can add this by including some contributions. Lets add in Absorption:

In [16]:
from taurex.contributions import AbsorptionContribution
tm.add_contribution(AbsorptionContribution())

And some CIA for good measure:

In [17]:
from taurex.contributions import CIAContribution
tm.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))

And some rayleigh

In [18]:
from taurex.contributions import RayleighContribution
tm.add_contribution(RayleighContribution())

Finally, putting it all together we **build** it to setup all the profiles

In [19]:
tm.build()

Thats it! Our transmission model is complete! We can now run it:

In [20]:
res = tm.model()
res

(array([  199.99326855,   200.00660143,   200.01993521, ...,
        33328.88933329, 33331.11125925, 33333.33333333], shape=(76744,)),
 array([0.01077477, 0.01100695, 0.01100066, ..., 0.01095715, 0.01095717,
        0.01095718], shape=(76744,)),
 array([[7.70334612e-252, 0.00000000e+000, 0.00000000e+000, ...,
         6.08239452e-006, 6.08239452e-006, 6.08239452e-006],
        [5.29593295e-101, 0.00000000e+000, 0.00000000e+000, ...,
         3.31717832e-121, 3.06389690e-121, 2.82988654e-121],
        [9.42629695e-041, 0.00000000e+000, 0.00000000e+000, ...,
         2.03631690e-076, 1.93657814e-076, 1.84169655e-076],
        ...,
        [9.99999987e-001, 9.99957070e-001, 9.99967553e-001, ...,
         9.98284128e-001, 9.98283628e-001, 9.98283128e-001],
        [9.99999995e-001, 9.99983962e-001, 9.99988123e-001, ...,
         9.99046539e-001, 9.99046261e-001, 9.99045984e-001],
        [9.99999998e-001, 9.99994793e-001, 9.99996224e-001, ...,
         9.99587902e-001, 9.99587782e-001, 9.9

Nice! The output has four components:

- The wavenumber grid
- The *native* flux
- The optical depth
- Any extra information

Now lets plot it! Lets see the chemistry of the atmosphere:

In [21]:
plt.figure()

for x,gasname in enumerate(tm.chemistry.activeGases):
    
    plt.plot(tm.chemistry.activeGasMixProfile[x],tm.pressureProfile/1e5,label=gasname)
for x,gasname in enumerate(tm.chemistry.inactiveGases):
    
    plt.plot(tm.chemistry.inactiveGasMixProfile[x],tm.pressureProfile/1e5,label=gasname)
plt.gca().invert_yaxis()
plt.yscale("log")
plt.xscale("log")
plt.legend()
plt.show()

<IPython.core.display.Javascript object>

Not interesting, but our chemistry model is pretty simple!

And now lets plot the flux

In [22]:
native_grid, rprs, tau, _ = res

full_fig = plt.figure()
plt.plot(np.log10(10000/native_grid),rprs)
plt.show()

<IPython.core.display.Javascript object>

Coool! But lets try binning. We will use a simple but fast binner *SimpleBinner*.

In [23]:
from taurex.binning import FluxBinner,SimpleBinner
binned_fig = plt.figure()


#Make a logarithmic grid
wngrid = np.sort(10000/np.logspace(-0.4,1.1,1000))
bn = SimpleBinner(wngrid=wngrid)

bin_wn, bin_rprs,_,_  = bn.bin_model(tm.model(wngrid=wngrid))

plt.plot(10000/bin_wn,bin_rprs)
plt.xscale('log')
plt.show()

<IPython.core.display.Javascript object>

Cool but the nice thing about TauREx is that you can alter any of the parameters and it will respond to it!


If any of the profiles are altered then model will respond to it. This means that parameters such as temperature and mix ratios and even contributions can be changed on the fly! Try this example out

In [24]:
wngrid = np.sort(10000/np.logspace(-0.4,1.1,1000))
fig = plt.figure()
ax = fig.add_subplot(1,1,1)

model, = ax.plot(np.log(10000/wngrid),bn.bin_model(tm.model(wngrid))[1])

def update_model(temperature=1500.0,h2o_mix=-4):
    guillot.equilTemperature = temperature
    tm['H2O'] = 10**h2o_mix
    model.set_ydata(bn.bin_model(tm.model(wngrid))[1])
    ax.relim();
    ax.autoscale_view()
    fig.canvas.draw()

interact(update_model,temperature=(800.0,2000.0,100),h2o_mix=(-7.0,-2.0,1));


<IPython.core.display.Javascript object>

interactive(children=(FloatSlider(value=1500.0, description='temperature', max=2000.0, min=800.0, step=100.0),…

You can do some crazy things! If you reuse the same profiles on other models. You essentially couple them! That means
you can have multiple model that all alter at the same time! We can see the equivalant Emission and Direct image spectrum like so:

In [25]:
from taurex.model import EmissionModel, DirectImageModel
em = EmissionModel(planet=planet,
                       temperature_profile=guillot,
                       chemistry=chemistry,
                       star=star,
                        atm_min_pressure=1e-0,
                       atm_max_pressure=1e6,
                       nlayers=30)
di = DirectImageModel(planet=planet,
                       temperature_profile=guillot,
                       chemistry=chemistry,
                       star=star,
                        atm_min_pressure=1e-0,
                       atm_max_pressure=1e6,
                       nlayers=30)

em.add_contribution(AbsorptionContribution())
em.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))
em.add_contribution(RayleighContribution())

di.add_contribution(AbsorptionContribution())
di.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))
di.add_contribution(RayleighContribution())

em.build()
di.build()

In [27]:
wngrid = np.sort(10000/np.logspace(-0.4,1.1,1000))

all_fig = plt.figure(figsize=(9,4))
tm_ax = all_fig.add_subplot(1,3,1)
em_ax = all_fig.add_subplot(1,3,2)
di_ax = all_fig.add_subplot(1,3,3)
model_tm, = tm_ax.plot(10000/wngrid,bn.bin_model(tm.model(wngrid))[1])
model_em, = em_ax.plot(10000/wngrid,bn.bin_model(em.model(wngrid))[1])
model_di, = di_ax.plot(10000/wngrid,bn.bin_model(di.model(wngrid))[1])
tm_ax.set_xscale('log')
em_ax.set_xscale('log')
di_ax.set_xscale('log')
tm_ax.set_title('Transmission')
em_ax.set_title('Emission')
di_ax.set_title('Direct Image')
tm_ax.set_xlabel('Wavelength (um)')
em_ax.set_xlabel('Wavelength (um)')
di_ax.set_xlabel('Wavelength (um)')

def update_model(temperature=1500.0,h2o_mix=-4):
    guillot.equilTemperature = temperature
    tm['H2O'] = 10**h2o_mix
    model_tm.set_ydata(bn.bin_model(tm.model(wngrid))[1])
    model_em.set_ydata(bn.bin_model(em.model(wngrid))[1])
    model_di.set_ydata(bn.bin_model(di.model(wngrid))[1])
    tm_ax.relim();
    tm_ax.autoscale_view()
    
    em_ax.relim();
    em_ax.autoscale_view()
    
    di_ax.relim();
    di_ax.autoscale_view()
    
    fig.canvas.draw()

interact(update_model,temperature=(800.0,2000.0,100),h2o_mix=(-7.0,-2.0,1));

<IPython.core.display.Javascript object>

interactive(children=(FloatSlider(value=1500.0, description='temperature', max=2000.0, min=800.0, step=100.0),…

## Retreivals

To see what parameters available for retrievals we can list them like so:

In [28]:
list(tm.fittingParameters.keys())

['planet_mass',
 'planet_radius',
 'planet_distance',
 'planet_sma',
 'distance',
 'atm_min_pressure',
 'atm_max_pressure',
 'T_irr',
 'kappa_irr',
 'kappa_v1',
 'kappa_v2',
 'alpha',
 'T_int_guillot',
 'H2O',
 'N2',
 'He_H2']

The thing is, this is *dynamic*, TauREx 3 figures out what can be fit based on whats in it. Lets throw an
isothermal profile instead:

In [29]:
from taurex.temperature import Isothermal

isothermal = Isothermal(T=1500.0)

tm = TransmissionModel(planet=planet,
                       temperature_profile=isothermal,
                       chemistry=chemistry,
                       star=star,
                       atm_min_pressure=1e-0,
                       atm_max_pressure=1e6,
                       nlayers=30)
tm.add_contribution(AbsorptionContribution())
tm.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))
tm.add_contribution(RayleighContribution())
tm.build()

In [30]:
tm.model()

(array([  199.99326855,   200.00660143,   200.01993521, ...,
        33328.88933329, 33331.11125925, 33333.33333333], shape=(76744,)),
 array([0.01079474, 0.01107104, 0.01106564, ..., 0.01101442, 0.01101444,
        0.01101446], shape=(76744,)),
 array([[1.70379056e-199, 0.00000000e+000, 0.00000000e+000, ...,
         4.27433912e-006, 4.27433912e-006, 4.27433912e-006],
        [5.03545273e-080, 0.00000000e+000, 0.00000000e+000, ...,
         2.19726092e-114, 2.03888267e-114, 1.89187742e-114],
        [2.31360907e-032, 0.00000000e+000, 0.00000000e+000, ...,
         4.31418216e-072, 4.11486650e-072, 3.92470299e-072],
        ...,
        [9.99999993e-001, 9.99962977e-001, 9.99963487e-001, ...,
         9.98502553e-001, 9.98502117e-001, 9.98501681e-001],
        [9.99999997e-001, 9.99984383e-001, 9.99983957e-001, ...,
         9.99167598e-001, 9.99167355e-001, 9.99167113e-001],
        [9.99999999e-001, 9.99994344e-001, 9.99993996e-001, ...,
         9.99640090e-001, 9.99639985e-001, 9.9

In [31]:
list(tm.fittingParameters.keys())

['planet_mass',
 'planet_radius',
 'planet_distance',
 'planet_sma',
 'distance',
 'atm_min_pressure',
 'atm_max_pressure',
 'T',
 'H2O',
 'N2',
 'He_H2']

Here we lost the Guillot parameters like *T_irr* but gained the Isothermal parameter *T*
We can also access them directly from the model using the brackets operator:

In [32]:
tm['T']

1500.0

And set them

In [33]:
tm['H2O']=1.2e-4

Now we'll need an observation. lets use *ObservedSpectrum* to load a text one from **examples/test_data.dat**:

In [34]:
from taurex.data.spectrum.observed import ObservedSpectrum
obs = ObservedSpectrum('../../examples/parfiles/quickstart.dat')

Convieniently we have a way of binning our native spectrum down to the observation by calling its *create_binner* method:

In [35]:
obin = obs.create_binner()

And we can now plot

In [36]:
plt.figure()
plt.errorbar(obs.wavelengthGrid,obs.spectrum,obs.errorBar,label='Obs')
plt.plot(obs.wavelengthGrid,obin.bin_model(tm.model(obs.wavenumberGrid))[1],label='TM')
plt.legend()
plt.show()

<IPython.core.display.Javascript object>

Ew not good! Lets try a retrieval! We have many optimizers to choose so lets create one using the inbuilt optimizer based on [nestle](http://kylebarbary.com/nestle/)

In [37]:
from taurex.optimizer.nestle import NestleOptimizer
opt = NestleOptimizer(num_live_points=50)

We need to tell it about our forward model and observation:

In [38]:
opt.set_model(tm)
opt.set_observed(obs)

Now lets enable which parameters to fit and their prior boundaries:

In [39]:
opt.enable_fit('planet_radius')
opt.enable_fit('T')
opt.set_boundary('T',[1000,2000])
opt.set_boundary('planet_radius',[0.8,2.1])

Now lets fit!!!!

In [41]:
start_time = time.time()
solution = opt.fit()
taurex.log.disableLogging()
print(f'finished after {time.time() - start_time}')

it=   700 logz=1877.144052niter: 701
ncall: 1234
nsamples: 751
logz: 1877.569 +/-  0.492
h: 12.127
finished after 135.05431699752808


Lets loop and plot each solution!

In [42]:
for solution,optimized_map,optimized_value,values in opt.get_solution():
    opt.update_model(optimized_map)
    plt.figure()
    plt.errorbar(obs.wavelengthGrid,obs.spectrum,obs.errorBar,label='Obs')
    plt.plot(obs.wavelengthGrid,obin.bin_model(tm.model(obs.wavenumberGrid))[1],label='TM')
    plt.legend()
    plt.show()

<IPython.core.display.Javascript object>

Nice!